In [2]:
import os
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [3]:
documents = [
    "Ahmed is a third-year AI and Data Science student.",
    "He is learning Retrieval-Augmented Generation to build a dental lecture assistant.",
    "A convolutional neural network is commonly used for image classification tasks.",
    "Passive eruption is related to the movement of the gingival margin after tooth eruption.",
    "Power BI is used to build dashboards and visualize business data."
]

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
doc_embeddings = model.encode(documents)
print(type(doc_embeddings))
print(doc_embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [6]:
question = "What Project is Ahmed building with RAG"

question_embedding = model.encode(question)
print(question_embedding.shape)

(384,)


In [7]:
def cosine_similarity(a, b):
    dot_product = np.dot(a,b)
    mag_a = np.linalg.norm(a)
    mag_b = np.linalg.norm(b)
    return dot_product / (mag_a * mag_b)

In [8]:
similarties = []

for doc, emb in zip(documents, doc_embeddings):
    similarity = cosine_similarity(emb, question_embedding)
    similarties.append(similarity)

result = pd.DataFrame({"document": documents, "similarity": similarties})

result = result.sort_values(by="similarity", ascending=True)
result

,document,similarity
3,Passive eruption is related to the movement of...,-0.013756
2,A convolutional neural network is commonly use...,0.030239
1,He is learning Retrieval-Augmented Generation ...,0.148563
4,Power BI is used to build dashboards and visua...,0.192862
0,Ahmed is a third-year AI and Data Science stud...,0.365891


In [19]:
def retrieve_top_k(question, documents, doc_embeddings, k=3):
    question_embedding = model.encode(question)

    scores = []
    for emb in doc_embeddings:
        score = cosine_similarity(question_embedding, emb)
        scores.append(score)

    top_indices = np.argsort(scores)[::-1][:k]

    retrieved = []
    for idx in top_indices:
        retrieved.append({
            "document": documents[idx],
            "score": scores[idx]
        })

    return retrieved

In [10]:
retrieve_top_k(question, documents, doc_embeddings)

[{'document': 'Ahmed is a third-year AI and Data Science student.',
  'score': np.float32(0.36589068)},
 {'document': 'Power BI is used to build dashboards and visualize business data.',
  'score': np.float32(0.19286232)},
 {'document': 'He is learning Retrieval-Augmented Generation to build a dental lecture assistant.',
  'score': np.float32(0.14856318)}]

In [11]:
question = "What is Ahmed Trying to build"

retrieved_chunks = retrieve_top_k(question=question,
                                  documents=documents,
                                  doc_embeddings=doc_embeddings,
                                  k=3)

for item in retrieved_chunks:
    print("Score: ", round(item["score"], 4))
    print("Chunk: ", item["document"])
    print("-" * 80)

Score:  0.5202
Chunk:  Ahmed is a third-year AI and Data Science student.
--------------------------------------------------------------------------------
Score:  0.1708
Chunk:  Power BI is used to build dashboards and visualize business data.
--------------------------------------------------------------------------------
Score:  0.1658
Chunk:  He is learning Retrieval-Augmented Generation to build a dental lecture assistant.
--------------------------------------------------------------------------------


In [12]:
def split_text_into_chunks(text, chunk_size=300, overlap=50):
    text = re.sub(r"/s+", " ", text)
    
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    
    return chunks

In [13]:
lecture_text = """
Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the teeth and covers the alveolar bone. Healthy gingiva is usually firm, pink, and stippled.

A convolutional neural network is a deep learning model commonly used for image classification. It uses convolutional layers to detect features such as edges, textures, and shapes.

Power BI is a business intelligence tool used to build dashboards and analyze business data.
"""

In [14]:
chunks = split_text_into_chunks(text=lecture_text)
for i, chunk in enumerate(chunks):
    print("Chunk ", i)
    print(chunk)
    print("-"  * 80)
    

Chunk  0

Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the oppos
--------------------------------------------------------------------------------
Chunk  1
ues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the tee
--------------------------------------------------------------------------------
Chunk  2
the part of the oral mucosa that surrounds the teeth and covers the alveolar bone. Healthy gingiva is usually firm, pink, and stippled.

A convolutional neural network is a deep learning model commonly used fo

In [16]:
chunk_embeddings = model.encode(chunks)

print(chunk_embeddings.shape)

(4, 384)


In [23]:
question = "What is passive eruption?"

retrieved_chunks = retrieve_top_k(
    question=question,
    documents=chunks,
    doc_embeddings=chunk_embeddings,
    k=2
)

for item in retrieved_chunks:
    print("Score:", round(item["score"], 4))
    print("Text:", item["document"])
    print("-" * 80)


Score: 0.6646
Text: ues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the tee
--------------------------------------------------------------------------------
Score: 0.5718
Text: 
Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the oppos
--------------------------------------------------------------------------------


In [24]:
from pypdf import PdfReader

def extract_text_from_pdf(path):
    pdf = PdfReader(path)
    
    text_pages = []
    for page_number, page in enumerate(pdf.pages):
        text = page.extract_text()
        
        if text:
            text_pages.append({
                "page": page_number + 1,
                "text": text
            })
    
    return text_pages

In [25]:
pdf_path = "sample_dental_lecture.pdf"

pages = extract_text_from_pdf(pdf_path)

print("num of pages", len(pages))
print(pages[0]["page"])
print(pages[0]["text"][:1000])

num of pages 4
1
Sample Dental Lecture PDF - Page 1
 Sample Dental Lecture for RAG Testing
This PDF is made for testing a simple Retrieval-Augmented Generation pipeline. It contains clean
selectable text, so pypdf should be able to extract it without OCR.
1. Tooth Eruption
Tooth eruption is the process by which a developing tooth moves from its position inside the jaw
bone into its functional position in the oral cavity. The process involves movement through bone
and soft tissue until the tooth reaches the occlusal plane.
Active eruption refers to the actual movement of the tooth toward the occlusal plane. It continues
until the tooth contacts the opposing tooth or reaches its correct functional position.
Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It
changes the relationship between the gingiva and the anatomical crown. Clinically, passive
eruption affects how much of the crown is visible in the mouth.
2. Gingiva
Gingiva is the p